In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import matplotlib.gridspec as gridspec

import seaborn as sns
import scipy.stats as stats
import math

from collections import defaultdict       # library to create dictionarie variables

from IPython.display import display, HTML, Markdown

In [3]:
def Print_3_txt(x,y,z,n_spaces,var_type):
    from IPython.display import display, HTML, Markdown, display_html

    length = len(x)
    if length > n_spaces: length = n_spaces
    display(HTML(f"<b> {x + ("&nbsp;" * (n_spaces-length))} :</b> <font size='3'>{y} </font> <font size='1'>{z}</font>"))
    
    if var_type == True:
        print('Tipo de variable:', type(y))
    if len(str(y))>0: print('')

In [1]:
def Plot_Input_func(Input_var_dict):
    import matplotlib.pyplot as plt
    import seaborn as sns
    import numpy as np
    
    fig, AX = plt.subplots(nrows = 4, ncols=4, figsize=(15,10), layout ='constrained')
    
    #cmap = matplotlib.cm.get_cmap('rocket')
    cmap = sns.color_palette('Set2', as_cmap=True)
    
    icolor = 0.1
    i_samples = 10000
    i_col = 0
    i_row = 0
    for key in Input_var_dict.keys():
        
        RBG = cmap(icolor)[0],cmap(icolor)[1],cmap(icolor)[2]
        icolor = icolor + 0.1
        
        i_mean, i_SD = Input_var_dict[key][2], Input_var_dict[key][3]
        i_Low_limit, i_High_limit = Input_var_dict[key][0], Input_var_dict[key][1]
        i_constant = Input_var_dict[key][4]
        i_x_Label, i_Title = Input_var_dict[key][6], Input_var_dict[key][6]
        i_Dist_Type = Input_var_dict[key][5]
    
        Generate_normal_and_Truncated(i_mean, i_SD, i_samples, i_Low_limit, i_High_limit, i_constant, i_x_Label, i_Title, i_col,i_row,AX,RBG,i_Dist_Type)
        
        i_col = i_col + 1
        if i_col > 3:
            i_col = 0
            i_row = 2

In [4]:
def Generate_normal_and_Truncated(V_mu,V_SD,V_samples,V_Low_limit,V_High_limit,V_Constant, V_x_Label, V_Title, n_col,n_row,AX,RBG, Dist_Type=None):
    import numpy as np
    import scipy.stats as stats
    
    n_bins = 30
    N_Samples = np.random.normal(loc = V_mu, scale = V_SD, size= V_samples) # normal distribution

    if Dist_Type == 'Constant':
        N_Samples = np.random.uniform(low = V_Constant, high = V_Constant, size = 10000)
        
    if Dist_Type == 'Uniform':
        N_Samples = np.random.uniform(low = V_Low_limit, high = V_High_limit, size = 10000)
        
    if Dist_Type == 'Normal':
        N_Samples = np.random.normal(loc = V_mu, scale = V_SD, size= V_samples) # normal distribution
    
    # Plot Normal Distribution
    AX[n_row,n_col].hist(N_Samples, bins=n_bins, density=True, align='mid', color=(RBG), alpha=1, edgecolor = "black")
    AX[n_row,n_col].set_xlabel(' ')
    AX[n_row,n_col].set_ylabel('Probability Density')
    AX[n_row,n_col].set_title(V_Title, weight='bold', fontsize = 18)
    x_axis = AX[n_row,n_col].set_xlim()

    
    ## generate data for truncated normal distribution
    
    NT_Samples = stats.truncnorm.rvs((V_Low_limit-V_mu)/V_SD,(V_High_limit-V_mu)/V_SD,loc=V_mu,scale=V_SD, size=V_samples)

    if Dist_Type == 'Constant':
        NT_Samples = np.random.uniform(low = V_Constant, high = V_Constant, size = 10000)
        
    if Dist_Type == 'Uniform':
        NT_Samples = np.random.uniform(low = V_Low_limit, high = V_High_limit, size = 10000)
        
    if Dist_Type == 'Normal':
        NT_Samples = stats.truncnorm.rvs((V_Low_limit-V_mu)/V_SD,(V_High_limit-V_mu)/V_SD,loc=V_mu,scale=V_SD, size=V_samples)
        
    ## Plot truncated Normal distribution
    AX[n_row+1,n_col].hist(NT_Samples, bins=n_bins, density=True, align='mid', color=RBG, alpha=0.4, edgecolor = "black")
    AX[n_row+1,n_col].set_xlabel(' ')
    AX[n_row+1,n_col].set_ylabel('PD')
    AX[n_row+1,n_col].set_title(V_Title + ' (TRUNCATED)')
    AX[n_row+1,n_col].set_xlim(x_axis)

### Function to calculate NPV IRR

In [ ]:
def Calc_Eco_Eval():

    Cash_Flow_df = pd.DataFrame()
    Cash_Flow_df = Prod_df.copy(deep=True)
    
    Cash_Flow_df.loc[:, 'OPEX'] = OPEX  * Cash_Flow_df['BOEs'] * Cash_Flow_df['Days']
    
    Cash_Flow_df.loc[:, 'Ab'] = 0
    Cash_Flow_df.iloc[-1, Cash_Flow_df.columns.get_loc('Ab')] = decommissioning
    
    Cash_Flow_df.loc[:, 'Headco Cost'] = -Headco_cost
    
    Cash_Flow_df.loc[:, 'Gas Price'] = BTU_scf * USD_MMBTU
    Cash_Flow_df.loc[:, 'Oil Price'] = Oil_Price
    
    Cash_Flow_df.loc[:, 'Gas Rev'] = Cash_Flow_df['Gas Sale'] * Cash_Flow_df['Gas Price']
    Cash_Flow_df.loc[:, 'Liquids Rev'] = (Cash_Flow_df['Oil']+Cash_Flow_df['Gasoline']) * Cash_Flow_df['Oil Price']
    
    Cash_Flow_df.loc[:, 'Total Rev USD_day'] = Cash_Flow_df['Gas Rev']+Cash_Flow_df['Liquids Rev']
    Cash_Flow_df.loc[:, 'Total Revenue year'] = Cash_Flow_df['Total Rev USD_day'] * Cash_Flow_df['Days']
    Cash_Flow_df.loc[:, 'Tax'] = Cash_Flow_df['Total Revenue year'] * -Gov_tax
    
    Cash_Flow_df.loc[:, 'EBITDA'] = Cash_Flow_df['Total Revenue year'] + Cash_Flow_df['OPEX'] + Cash_Flow_df['CAPEX'] 
    
    Cash_Flow_df.loc[:, 'Operating Cash Flow'] = Cash_Flow_df['EBITDA'] + Cash_Flow_df['Tax'] + Cash_Flow_df['Ab'] 
    
    Cash_Flow_df.loc[:, 'Free Cash Flow'] = Cash_Flow_df['Operating Cash Flow'] + Cash_Flow_df['Ab'] 
    
    pd.set_option('display.float_format', '{:.2f}'.format)
    
    
    
    discount_rate = 0.10  # 10% cost of capital
    cash_flows_L = Cash_Flow_df['Free Cash Flow'].tolist()
    #print(cash_flows_L)
    
    # Compute the NPV
    # The function automatically treats cash_flows[0] as Year 0 (undiscounted)
    npv_result = npf.npv(discount_rate, cash_flows_L)
    print('---------------------------')
    print(f"Project NPV: ${npv_result:,.2f}") # Display the formatted result
    
    # Calculate IRR
    irr = pyxirr.irr(cash_flows_L)
    print(f"Internal Rate of Return: {irr:.3f}")
    
    Cash_Flow_df.head(4)
    display(Cash_Flow_df)
    
    specific_sum = Cash_Flow_df[['CAPEX', 'Free Cash Flow']].sum()
    display(specific_sum)
